In [1]:
#!/usr/bin/env python
# coding: utf-8

# =============================================================================
# 导入库，加载数据和CV设置
# =============================================================================

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import json
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

Path('models/xgb').mkdir(parents=True, exist_ok=True)

df = pd.read_csv('data/full_dataset.csv')

with open('data/cv_setup.pkl', 'rb') as f:
    cv_setup = pickle.load(f)
with open('data/cv_setup_inner.pkl', 'rb') as f:
    cv_setup_inner = pickle.load(f)

feature_cols = cv_setup['feature_cols']
target_col   = cv_setup['target_col']
outer_folds  = cv_setup['outer_folds']

X = df[feature_cols].values
y = df[target_col].values

# 超参数搜索空间
param_grid = {
    'n_estimators':    [100, 200, 300],
    'learning_rate':   [0.01, 0.1, 0.2],
    'max_depth':       [3, 6, 9],
    'subsample':       [0.8, 1.0],
    'colsample_bytree':[0.8, 1.0]
}

EDGE_THRESHOLD   = 0.1
ATOM_TYPE_COLS   = ['T_O', 'T_C', 'T_Ti1', 'T_Ti2']
ATOM_TYPE_LABELS = ['O', 'C', 'Ti_inner', 'Ti_outer']

# =============================================================================
# 辅助函数：计算指标，分类评估，电中性校正
# =============================================================================

def calculate_metrics(y_true, y_pred):
    return {
        'mae':       float(mean_absolute_error(y_true, y_pred)),
        'rmse':      float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'r2':        float(r2_score(y_true, y_pred)),
        'n_samples': int(len(y_true))
    }

def classify_and_evaluate(y_true, y_pred, df_subset):
    results = {}
    results['overall'] = calculate_metrics(y_true, y_pred)

    edge_mask     = df_subset['D_E'].values < EDGE_THRESHOLD
    interior_mask = ~edge_mask

    results['edge']     = calculate_metrics(y_true[edge_mask],     y_pred[edge_mask])     if edge_mask.sum()     > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}
    results['interior'] = calculate_metrics(y_true[interior_mask], y_pred[interior_mask]) if interior_mask.sum() > 0 else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}

    results['by_atom_type'] = {}
    for col, label in zip(ATOM_TYPE_COLS, ATOM_TYPE_LABELS):
        mask = df_subset[col].values == 1
        results['by_atom_type'][label] = calculate_metrics(y_true[mask], y_pred[mask]) if mask.sum() > 0 \
            else {'mae': np.nan, 'rmse': np.nan, 'r2': np.nan, 'n_samples': 0}
    return results

def apply_charge_neutrality_correction(df_predictions):
    """按电荷绝对值加权，对每个结构应用电中性校正"""
    df_corrected = df_predictions.copy()
    df_corrected['y_pred_corrected']  = 0.0
    df_corrected['correction_amount'] = 0.0
    df_corrected['correction_weight'] = 0.0
    df_corrected['Q_total_before']    = 0.0

    for structure_id in df_corrected['structure_id'].unique():
        mask    = df_corrected['structure_id'] == structure_id
        charges = df_corrected.loc[mask, 'y_pred'].values
        Q_total = charges.sum()

        abs_charges     = np.abs(charges)
        sum_abs_charges = abs_charges.sum()
        weights = abs_charges / sum_abs_charges if sum_abs_charges > 1e-10 \
                  else np.ones(len(charges)) / len(charges)

        corrections = weights * Q_total
        df_corrected.loc[mask, 'y_pred_corrected']  = charges - corrections
        df_corrected.loc[mask, 'correction_amount'] = corrections
        df_corrected.loc[mask, 'correction_weight'] = weights
        df_corrected.loc[mask, 'Q_total_before']    = Q_total

    df_corrected['error_corrected']     = df_corrected['y_pred_corrected'] - df_corrected['y_true']
    df_corrected['abs_error_corrected'] = np.abs(df_corrected['error_corrected'])
    return df_corrected

def _get_xgb_importance(model, n_features, importance_type='gain'):
    """从XGBoost booster提取归一化特征重要性（按列索引 f0,f1,...）"""
    booster   = model.get_booster()
    imp_dict  = booster.get_score(importance_type=importance_type)
    imp_array = np.zeros(n_features)
    for i in range(n_features):
        imp_array[i] = imp_dict.get(f'f{i}', 0)
    if imp_array.sum() > 0:
        imp_array = imp_array / imp_array.sum()
    return imp_array

# =============================================================================
# 嵌套交叉验证主循环
# =============================================================================

outer_results              = []
best_models                = []
all_train_predictions      = []
all_test_predictions       = []
feature_importances_gain   = []   # XGBoost特有：gain重要性
feature_importances_weight = []   # XGBoost特有：weight重要性
classification_results     = {'fold_wise': {}, 'averaged': {}}

start_time = time.time()

for fold_idx, outer_fold in enumerate(outer_folds):
    train_idx = outer_fold['train_atoms_idx']
    test_idx  = outer_fold['test_atoms_idx']

    X_train = X[train_idx];  y_train = y[train_idx]
    X_test  = X[test_idx];   y_test  = y[test_idx]

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)

    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    inner_folds      = cv_setup_inner[fold_idx]['inner_folds']
    inner_cv_indices = []
    for inner_fold in inner_folds:
        inner_train_set = set(inner_fold['inner_train_structures'])
        inner_val_set   = set(inner_fold['inner_val_structures'])
        struct_ids = df.iloc[train_idx]['structure_id'].values
        inner_cv_indices.append((
            np.where([s in inner_train_set for s in struct_ids])[0],
            np.where([s in inner_val_set   for s in struct_ids])[0]
        ))

    grid_search = GridSearchCV(
        estimator=XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist'),
        param_grid=param_grid,
        cv=inner_cv_indices,
        scoring='neg_mean_squared_error',
        n_jobs=1,   # XGBoost已内部并行
        verbose=0
    )
    grid_search.fit(X_train_scaled, y_train)
    best_params = grid_search.best_params_

    best_model = XGBRegressor(**best_params, random_state=42, n_jobs=-1, tree_method='hist')
    best_model.fit(X_train_scaled, y_train)

    # 记录各折特征重要性
    feature_importances_gain.append({'fold': fold_idx+1,
        'importances': _get_xgb_importance(best_model, len(feature_cols), 'gain').tolist()})
    feature_importances_weight.append({'fold': fold_idx+1,
        'importances': _get_xgb_importance(best_model, len(feature_cols), 'weight').tolist()})

    y_train_pred = best_model.predict(X_train_scaled)
    y_test_pred  = best_model.predict(X_test_scaled)

    train_class_results = classify_and_evaluate(y_train, y_train_pred, df_train)
    test_class_results  = classify_and_evaluate(y_test,  y_test_pred,  df_test)

    print(f"折叠 {fold_idx+1} | 最优参数: {best_params} | "
          f"训练MAE: {train_class_results['overall']['mae']:.5f} | "
          f"测试MAE: {test_class_results['overall']['mae']:.5f}")

    fold_result = {
        'fold': fold_idx + 1,
        'best_params': best_params,
        'train_metrics': train_class_results,
        'test_metrics':  test_class_results,
        'n_train': len(train_idx),
        'n_test':  len(test_idx),
        'train_structures': outer_fold['train_structures'],
        'test_structures':  outer_fold['test_structures']
    }
    outer_results.append(fold_result)
    best_models.append({'fold': fold_idx+1, 'model': best_model,
                        'scaler': scaler, 'best_params': best_params})

    classification_results['fold_wise'][f'fold_{fold_idx+1}'] = {
        'train': train_class_results,
        'test':  test_class_results
    }

    for i in range(len(train_idx)):
        all_train_predictions.append({
            'fold': fold_idx+1, 'atom_index': int(train_idx[i]),
            'structure_id': df_train.iloc[i]['structure_id'],
            'D_E': float(df_train.iloc[i]['D_E']),
            'T_O': int(df_train.iloc[i]['T_O']), 'T_C': int(df_train.iloc[i]['T_C']),
            'T_Ti1': int(df_train.iloc[i]['T_Ti1']), 'T_Ti2': int(df_train.iloc[i]['T_Ti2']),
            'y_true': float(y_train[i]), 'y_pred': float(y_train_pred[i]),
            'error': float(y_train_pred[i]-y_train[i]),
            'abs_error': float(abs(y_train_pred[i]-y_train[i]))
        })
    for i in range(len(test_idx)):
        all_test_predictions.append({
            'fold': fold_idx+1, 'atom_index': int(test_idx[i]),
            'structure_id': df_test.iloc[i]['structure_id'],
            'D_E': float(df_test.iloc[i]['D_E']),
            'T_O': int(df_test.iloc[i]['T_O']), 'T_C': int(df_test.iloc[i]['T_C']),
            'T_Ti1': int(df_test.iloc[i]['T_Ti1']), 'T_Ti2': int(df_test.iloc[i]['T_Ti2']),
            'y_true': float(y_test[i]), 'y_pred': float(y_test_pred[i]),
            'error': float(y_test_pred[i]-y_test[i]),
            'abs_error': float(abs(y_test_pred[i]-y_test[i]))
        })

elapsed_time = time.time() - start_time
print(f"\n嵌套CV完成，耗时 {elapsed_time:.1f} 秒 ({elapsed_time/60:.1f} 分钟)")

# =============================================================================
# 特征重要性汇总，计算平均性能，应用电中性校正，保存所有结果
# =============================================================================

# 特征重要性汇总
mean_importances_gain   = np.mean([fi['importances'] for fi in feature_importances_gain],   axis=0)
std_importances_gain    = np.std( [fi['importances'] for fi in feature_importances_gain],   axis=0)
mean_importances_weight = np.mean([fi['importances'] for fi in feature_importances_weight], axis=0)
std_importances_weight  = np.std( [fi['importances'] for fi in feature_importances_weight], axis=0)

importance_df_gain   = pd.DataFrame({'feature': feature_cols,
    'importance': mean_importances_gain,   'std': std_importances_gain}).sort_values('importance', ascending=False)
importance_df_weight = pd.DataFrame({'feature': feature_cols,
    'importance': mean_importances_weight, 'std': std_importances_weight}).sort_values('importance', ascending=False)

importance_df_gain.to_csv('models/xgb/xgb_feature_importance_gain.csv',   index=False)
importance_df_weight.to_csv('models/xgb/xgb_feature_importance_weight.csv', index=False)

print("\nTop 10 最重要特征 (Gain):")
print(importance_df_gain.head(10).to_string(index=False))

def calculate_averaged_metrics(outer_results, dataset_type='test'):
    key      = f'{dataset_type}_metrics'
    averaged = {}
    for category in ['overall', 'edge', 'interior']:
        maes  = [r[key][category]['mae']  for r in outer_results]
        rmses = [r[key][category]['rmse'] for r in outer_results]
        r2s   = [r[key][category]['r2']   for r in outer_results]
        ns    = [r[key][category]['n_samples'] for r in outer_results]
        averaged[category] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }
    averaged['by_atom_type'] = {}
    for label in ATOM_TYPE_LABELS:
        maes  = [r[key]['by_atom_type'][label]['mae']  for r in outer_results]
        rmses = [r[key]['by_atom_type'][label]['rmse'] for r in outer_results]
        r2s   = [r[key]['by_atom_type'][label]['r2']   for r in outer_results]
        ns    = [r[key]['by_atom_type'][label]['n_samples'] for r in outer_results]
        averaged['by_atom_type'][label] = {
            'mae_mean':  float(np.mean(maes)),  'mae_std':  float(np.std(maes)),
            'rmse_mean': float(np.mean(rmses)), 'rmse_std': float(np.std(rmses)),
            'r2_mean':   float(np.mean(r2s)),   'r2_std':   float(np.std(r2s)),
            'n_samples_mean': float(np.mean(ns))
        }
    return averaged

train_averaged = calculate_averaged_metrics(outer_results, 'train')
test_averaged  = calculate_averaged_metrics(outer_results, 'test')
classification_results['averaged'] = {'train': train_averaged, 'test': test_averaged}

print("\n5折CV平均性能（测试集，校正前）:")
for cat in ['overall', 'edge', 'interior']:
    m = test_averaged[cat]
    print(f"  {cat:10s} - MAE: {m['mae_mean']:.5f} ± {m['mae_std']:.5f}  "
          f"RMSE: {m['rmse_mean']:.5f} ± {m['rmse_std']:.5f}  "
          f"R²: {m['r2_mean']:.6f} ± {m['r2_std']:.6f}")

# 电中性校正
df_train_all = pd.DataFrame(all_train_predictions)
df_test_all  = pd.DataFrame(all_test_predictions)

df_train_all_corrected = apply_charge_neutrality_correction(df_train_all)
df_test_all_corrected  = apply_charge_neutrality_correction(df_test_all)

# 校正后逐折计算指标
outer_results_corrected          = []
classification_results_corrected = {'fold_wise': {}, 'averaged': {}}

for fold_idx in range(len(outer_folds)):
    fold_num = fold_idx + 1
    df_train_fold_corr = df_train_all_corrected[df_train_all_corrected['fold'] == fold_num]
    df_test_fold_corr  = df_test_all_corrected[ df_test_all_corrected['fold']  == fold_num]

    train_class_corr = classify_and_evaluate(
        df_train_fold_corr['y_true'].values, df_train_fold_corr['y_pred_corrected'].values, df_train_fold_corr)
    test_class_corr  = classify_and_evaluate(
        df_test_fold_corr['y_true'].values,  df_test_fold_corr['y_pred_corrected'].values,  df_test_fold_corr)

    outer_results_corrected.append({'fold': fold_num, 'train_metrics': train_class_corr,
                                    'test_metrics': test_class_corr,
                                    'n_train': len(df_train_fold_corr), 'n_test': len(df_test_fold_corr)})
    classification_results_corrected['fold_wise'][f'fold_{fold_num}'] = {
        'train': train_class_corr, 'test': test_class_corr}

train_averaged_corrected = calculate_averaged_metrics(outer_results_corrected, 'train')
test_averaged_corrected  = calculate_averaged_metrics(outer_results_corrected, 'test')
classification_results_corrected['averaged'] = {'train': train_averaged_corrected,
                                                 'test':  test_averaged_corrected}

print("\n5折CV平均性能（测试集，校正后）:")
for cat in ['overall', 'edge', 'interior']:
    m = test_averaged_corrected[cat]
    print(f"  {cat:10s} - MAE: {m['mae_mean']:.5f} ± {m['mae_std']:.5f}  "
          f"RMSE: {m['rmse_mean']:.5f} ± {m['rmse_std']:.5f}  "
          f"R²: {m['r2_mean']:.6f} ± {m['r2_std']:.6f}")

# 校正效果统计
test_Q_before = df_test_all_corrected.groupby('structure_id')['Q_total_before'].first()
test_Q_after  = df_test_all_corrected.groupby('structure_id')['y_pred_corrected'].sum()
print(f"\n测试集校正前平均|Q_total|: {np.abs(test_Q_before).mean():.6f} e")
print(f"测试集校正后平均|Q_total|: {np.abs(test_Q_after).mean():.2e} e")

# 最终模型
best_fold_idx = np.argmin([r['test_metrics']['overall']['mae'] for r in outer_results])
final_params  = outer_results[best_fold_idx]['best_params']
print(f"\n最终模型参数（折叠{best_fold_idx+1}）: {final_params}")

scaler_final   = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)
final_model    = XGBRegressor(**final_params, random_state=42, n_jobs=-1, tree_method='hist')
final_model.fit(X_scaled_final, y)
final_importances_gain = _get_xgb_importance(final_model, len(feature_cols), 'gain')

# 保存预测CSV
for fold_num in range(1, 6):
    df_train_all[df_train_all['fold'] == fold_num].to_csv(
        f'models/xgb/fold_{fold_num}_train_predictions.csv', index=False)
    df_test_all[df_test_all['fold'] == fold_num].to_csv(
        f'models/xgb/fold_{fold_num}_test_predictions.csv', index=False)
    df_train_all_corrected[df_train_all_corrected['fold'] == fold_num].to_csv(
        f'models/xgb/fold_{fold_num}_train_predictions_corrected.csv', index=False)
    df_test_all_corrected[df_test_all_corrected['fold'] == fold_num].to_csv(
        f'models/xgb/fold_{fold_num}_test_predictions_corrected.csv', index=False)

# 保存分类评分CSV
def build_classification_rows(classification_results, atom_type_labels):
    rows = []
    for fold_num in range(1, 6):
        fold_data = classification_results['fold_wise'][f'fold_{fold_num}']
        for dataset in ['train', 'test']:
            for category in ['overall', 'edge', 'interior']:
                d = fold_data[dataset][category]
                rows.append({'fold': fold_num, 'dataset': dataset,
                             'category': category, 'subcategory': 'all',
                             'mae': d['mae'], 'rmse': d['rmse'], 'r2': d['r2'], 'n_samples': d['n_samples']})
            for label in atom_type_labels:
                d = fold_data[dataset]['by_atom_type'][label]
                rows.append({'fold': fold_num, 'dataset': dataset,
                             'category': 'atom_type', 'subcategory': label,
                             'mae': d['mae'], 'rmse': d['rmse'], 'r2': d['r2'], 'n_samples': d['n_samples']})
    return rows

pd.DataFrame(build_classification_rows(classification_results, ATOM_TYPE_LABELS)).to_csv(
    'models/xgb/classification_scores.csv', index=False)
pd.DataFrame(build_classification_rows(classification_results_corrected, ATOM_TYPE_LABELS)).to_csv(
    'models/xgb/classification_scores_corrected.csv', index=False)

# 保存校正前后对比CSV
comparison_rows = []
for category_name, category_key in [('Overall','overall'), ('Edge','edge'), ('Interior','interior')]:
    mb = test_averaged[category_key]['mae_mean']
    ma = test_averaged_corrected[category_key]['mae_mean']
    comparison_rows.append({'category': category_name, 'subcategory': 'all',
        'mae_before': mb, 'mae_after': ma, 'mae_improvement_percent': (mb-ma)/mb*100,
        'mae_std_before': test_averaged[category_key]['mae_std'],
        'mae_std_after':  test_averaged_corrected[category_key]['mae_std']})
for label in ATOM_TYPE_LABELS:
    mb = test_averaged['by_atom_type'][label]['mae_mean']
    ma = test_averaged_corrected['by_atom_type'][label]['mae_mean']
    comparison_rows.append({'category': 'Atom_Type', 'subcategory': label,
        'mae_before': mb, 'mae_after': ma, 'mae_improvement_percent': (mb-ma)/mb*100,
        'mae_std_before': test_averaged['by_atom_type'][label]['mae_std'],
        'mae_std_after':  test_averaged_corrected['by_atom_type'][label]['mae_std']})
pd.DataFrame(comparison_rows).to_csv('models/xgb/correction_comparison.csv', index=False)

# 保存校正量统计CSV
edge_mask_all     = df_test_all_corrected['D_E'] < EDGE_THRESHOLD
interior_mask_all = ~edge_mask_all
correction_stats_rows = [{'category': 'overall', 'subcategory': 'all',
    'mean_correction':     float(df_test_all_corrected['correction_amount'].abs().mean()),
    'edge_correction':     float(df_test_all_corrected.loc[edge_mask_all,     'correction_amount'].abs().mean()),
    'interior_correction': float(df_test_all_corrected.loc[interior_mask_all, 'correction_amount'].abs().mean())}]
for col, label in zip(ATOM_TYPE_COLS, ATOM_TYPE_LABELS):
    atom_mask          = df_test_all_corrected[col] == 1
    atom_edge_mask     = atom_mask & edge_mask_all
    atom_interior_mask = atom_mask & interior_mask_all
    correction_stats_rows.append({'category': 'atom_type', 'subcategory': label,
        'mean_correction':     float(df_test_all_corrected.loc[atom_mask,          'correction_amount'].abs().mean()),
        'edge_correction':     float(df_test_all_corrected.loc[atom_edge_mask,     'correction_amount'].abs().mean()) if atom_edge_mask.sum()     > 0 else float('nan'),
        'interior_correction': float(df_test_all_corrected.loc[atom_interior_mask, 'correction_amount'].abs().mean()) if atom_interior_mask.sum() > 0 else float('nan')})
pd.DataFrame(correction_stats_rows).to_csv('models/xgb/correction_stats.csv', index=False)

# 保存模型文件
model_package = {
    'model': final_model, 'scaler': scaler_final,
    'feature_cols': feature_cols, 'target_col': target_col,
    'best_params': final_params,
    'feature_importances_gain': final_importances_gain.tolist(),
    'cv_performance': {'train': train_averaged, 'test': test_averaged},
    'cv_performance_corrected': {'train': train_averaged_corrected, 'test': test_averaged_corrected},
    'edge_threshold': EDGE_THRESHOLD,
    'atom_type_labels': ATOM_TYPE_LABELS
}
with open('models/xgb/xgb_final_model.pkl', 'wb') as f:
    pickle.dump(model_package, f)

with open('models/xgb/xgb_all_folds.pkl', 'wb') as f:
    pickle.dump(best_models, f)

results_dict = {
    'model_name': 'XGBoost Regressor',
    'n_folds': len(outer_folds),
    'n_features': len(feature_cols),
    'training_time_seconds': float(elapsed_time),
    'edge_threshold': EDGE_THRESHOLD,
    'param_grid': param_grid,
    'cv_results': {'train': train_averaged, 'test': test_averaged},
    'cv_results_corrected': {'train': train_averaged_corrected, 'test': test_averaged_corrected},
    'fold_details': outer_results,
    'feature_importances': {
        'features':               feature_cols,
        'mean_importance_gain':   mean_importances_gain.tolist(),
        'std_importance_gain':    std_importances_gain.tolist(),
        'mean_importance_weight': mean_importances_weight.tolist(),
        'std_importance_weight':  std_importances_weight.tolist(),
        'top_10_gain':            importance_df_gain.head(10).to_dict('records'),
        'top_10_weight':          importance_df_weight.head(10).to_dict('records')
    },
    'final_model': {'params': final_params}
}
with open('models/xgb/xgb_results.json', 'w') as f:
    json.dump(results_dict, f, indent=4)

print("\n已保存: 预测CSV(校正前/后), classification_scores.csv, "
      "classification_scores_corrected.csv, correction_comparison.csv, "
      "correction_stats.csv, xgb_feature_importance_gain/weight.csv, "
      "xgb_final_model.pkl, xgb_all_folds.pkl, xgb_results.json")

折叠 1 | 最优参数: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 6, 'n_estimators': 300, 'subsample': 1.0} | 训练MAE: 0.00245 | 测试MAE: 0.00534
折叠 2 | 最优参数: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 300, 'subsample': 1.0} | 训练MAE: 0.00205 | 测试MAE: 0.00452
折叠 3 | 最优参数: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 9, 'n_estimators': 100, 'subsample': 1.0} | 训练MAE: 0.00188 | 测试MAE: 0.00470
折叠 4 | 最优参数: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 300, 'subsample': 1.0} | 训练MAE: 0.00213 | 测试MAE: 0.00460
折叠 5 | 最优参数: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 1.0} | 训练MAE: 0.00298 | 测试MAE: 0.00550

嵌套CV完成，耗时 945.4 秒 (15.8 分钟)

Top 10 最重要特征 (Gain):
feature  importance      std
    Rad    0.593063 0.224313
  N_Ti2    0.381732 0.224323
    N_O    0.017540 0.000492
     EN    0.005358 0.006303
  T_Ti1    0.001915 0.000972
    T_C    0.000101 0.000